# DRL Assignment 3: Meta and Transfer Learning

**Setup for Google Colab with GPU acceleration**

Before running:
1. Go to Runtime → Change runtime type → Select **GPU** (T4, A100, or V100)
2. For best performance, select **A100** if available with your processor units

In [1]:
# Detect platform (Colab vs Kaggle vs other)
import os
try:
    IN_COLAB = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())
except:
    IN_COLAB = False
IN_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f"Running on: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Other'}")

# Clone repository (force fresh clone to ensure latest code)
import shutil
if os.path.exists('DRL-ass3'):
    shutil.rmtree('DRL-ass3')
!git clone https://github.com/omereliy/DRL-ass3.git
%cd DRL-ass3

# Uninstall old gym to avoid NumPy 2.0 conflicts
%pip uninstall -y gym 2>/dev/null || true

# Install dependencies with pinned versions (gymnasium>=0.30.0 for NumPy 2.0 compatibility)
%pip install -q "gymnasium>=0.30.0" torch tensorboard numpy

Running on: Kaggle
Cloning into 'DRL-ass3'...
remote: Enumerating objects: 150, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 150 (delta 75), reused 107 (delta 37), pack-reused 0 (from 0)
Receiving objects: 100% (150/150), 15.29 MiB | 34.63 MiB/s, done.
Resolving deltas: 100% (75/75), done.
/kaggle/working/DRL-ass3
Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 13.5 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, which is not installed.
stable-baselines3 2.1.0 requires gymnasium<0.30,>=0.28.1, but you have gymnasium 1.2.3 which is incompatible.
k

In [1]:
# Check GPU availability
!nvidia-smi

Wed Jan 21 10:17:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Verify GPU is available for PyTorch
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.8.0+cu126
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.83 GB


In [4]:
# Import modules with platform-agnostic path
import sys
import os

# Set base path based on platform
if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    base_path = '/kaggle/working/DRL-ass3'
else:
    base_path = '/content/DRL-ass3'

sys.path.insert(0, base_path)

from src.utils import TrainingConfig, DEVICE
from src.actor_critic import train_individual_network
from src.fine_tuning import run_section2_experiments
from src.progressive_networks import run_section3_experiments

print(f"Using device: {DEVICE}")

2026-01-21 10:17:24.833881: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768990644.855161     154 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768990644.861549     154 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768990644.878476     154 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768990644.878494     154 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768990644.878497     154 computation_placer.cc:177] computation placer alr

Using device: cuda


## Training Configuration

**Skip training options:** Set these to `True` to use pre-trained models from the repository instead of training from scratch. This is useful when you only want to run Section 3 with the fixed progressive networks.

**Note:** The repo includes pre-trained models, so you can skip Sections 1 & 2 and only run Section 3.

In [5]:
# =============================================================================
# SKIP TRAINING OPTIONS
# =============================================================================
# Set to True to skip training and use pre-trained models from the repository.
# This is useful when you only want to run Section 3 with the fixed progressive networks.

SKIP_SECTION_1 = True  # Skip individual network training (CartPole, Acrobot, MountainCar)
SKIP_SECTION_2 = True  # Skip fine-tuning experiments
SKIP_SECTION_3 = False  # Skip progressive networks (set False to test the fix!)

# Print configuration
print("Training Configuration:")
print(f"  SKIP_SECTION_1: {SKIP_SECTION_1}")
print(f"  SKIP_SECTION_2: {SKIP_SECTION_2}")
print(f"  SKIP_SECTION_3: {SKIP_SECTION_3}")

if SKIP_SECTION_1 or SKIP_SECTION_2:
    print("\nNote: Skipped sections will use pre-trained models from the repository.")

Training Configuration:
  SKIP_SECTION_1: True
  SKIP_SECTION_2: True
  SKIP_SECTION_3: False

Note: Skipped sections will use pre-trained models from the repository.


In [6]:
# Optimized configuration for faster training
fast_config = TrainingConfig(
    gamma=0.99,
    lr_actor=3e-3,      # Higher LR for faster convergence
    lr_critic=3e-3,
    hidden_dim=256,     # Larger network - GPUs handle this well
    max_episodes=1500,  # Usually converges before this
    max_steps=500,
    entropy_coef=0.01,
    value_loss_coef=0.5,
    log_interval=50,    # Less frequent logging
    save_interval=200,
    seed=42
)

## Section 1: Train Individual Networks

Train actor-critic networks for:
- CartPole-v1
- Acrobot-v1  
- MountainCarContinuous-v0

In [7]:
%%time
# Train or skip CartPole
from src.utils import TrainingStats

if SKIP_SECTION_1:
    print("="*60)
    print("SKIPPING Section 1: Using pre-trained models from repository")
    print("="*60)
    # Create placeholder stats (models are already in repo)
    cartpole_stats = TrainingStats()
    cartpole_stats.final_avg_reward = 490.84  # From previous run
    cartpole_stats.total_episodes = 958
    cartpole_stats.training_time = 0
    cartpole_stats.convergence_episode = 857
    print("CartPole-v1: Using pre-trained model")
else:
    print("="*60)
    print("Training CartPole-v1")
    print("="*60)
    cartpole_stats = train_individual_network("CartPole-v1", fast_config)

SKIPPING Section 1: Using pre-trained models from repository
CartPole-v1: Using pre-trained model
CPU times: user 100 µs, sys: 0 ns, total: 100 µs
Wall time: 97 µs


In [8]:
%%time
# Train or skip Acrobot
if SKIP_SECTION_1:
    acrobot_stats = TrainingStats()
    acrobot_stats.final_avg_reward = -88.00
    acrobot_stats.total_episodes = 375
    acrobot_stats.training_time = 0
    acrobot_stats.convergence_episode = 274
    print("Acrobot-v1: Using pre-trained model")
else:
    print("="*60)
    print("Training Acrobot-v1")
    print("="*60)
    acrobot_stats = train_individual_network("Acrobot-v1", fast_config)

Acrobot-v1: Using pre-trained model
CPU times: user 65 µs, sys: 0 ns, total: 65 µs
Wall time: 61.3 µs


In [9]:
%%time
# Train or skip MountainCar
if SKIP_SECTION_1:
    mountaincar_stats = TrainingStats()
    mountaincar_stats.final_avg_reward = -9.42
    mountaincar_stats.total_episodes = 1500
    mountaincar_stats.training_time = 0
    mountaincar_stats.convergence_episode = None
    print("MountainCarContinuous-v0: Using pre-trained model")
else:
    print("="*60)
    print("Training MountainCarContinuous-v0")
    print("="*60)
    mountaincar_stats = train_individual_network("MountainCarContinuous-v0", fast_config)

MountainCarContinuous-v0: Using pre-trained model
CPU times: user 58 µs, sys: 6 µs, total: 64 µs
Wall time: 62.2 µs


In [10]:
# Section 1 Summary
print("\n" + "="*70)
print("SECTION 1 RESULTS")
print("="*70)
section1_results = {
    "CartPole-v1": cartpole_stats,
    "Acrobot-v1": acrobot_stats,
    "MountainCarContinuous-v0": mountaincar_stats
}

print(f"{'Environment':<30} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*80)
for env, stats in section1_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{env:<30} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")


SECTION 1 RESULTS
Environment                    Episodes     Time (s)     Converged    Avg Reward     
--------------------------------------------------------------------------------
CartPole-v1                    958          0.00         857          490.84         
Acrobot-v1                     375          0.00         274          -88.00         
MountainCarContinuous-v0       1500         0.00         N/A          -9.42          


## Section 2: Fine-tuning

Transfer learning via fine-tuning:
1. Acrobot → CartPole
2. CartPole → MountainCar

In [11]:
%%time
# Run or skip Section 2 experiments
if SKIP_SECTION_2:
    print("="*60)
    print("SKIPPING Section 2: Using pre-trained fine-tuned models")
    print("="*60)
    # Create placeholder stats
    section2_results = {
        'acrobot_to_cartpole': TrainingStats(),
        'cartpole_to_mountaincar': TrainingStats()
    }
    section2_results['acrobot_to_cartpole'].final_avg_reward = 484.87
    section2_results['acrobot_to_cartpole'].total_episodes = 623
    section2_results['acrobot_to_cartpole'].training_time = 0
    section2_results['acrobot_to_cartpole'].convergence_episode = 522
    
    section2_results['cartpole_to_mountaincar'].final_avg_reward = -9.41
    section2_results['cartpole_to_mountaincar'].total_episodes = 1500
    section2_results['cartpole_to_mountaincar'].training_time = 0
    section2_results['cartpole_to_mountaincar'].convergence_episode = None
    print("Using pre-trained fine-tuned models from repository")
else:
    section2_results = run_section2_experiments(fast_config)

SKIPPING Section 2: Using pre-trained fine-tuned models
Using pre-trained fine-tuned models from repository
CPU times: user 68 µs, sys: 7 µs, total: 75 µs
Wall time: 71.8 µs


In [12]:
# Section 2 Summary
print("\n" + "="*70)
print("SECTION 2 RESULTS - Fine-tuning")
print("="*70)
print(f"{'Transfer':<35} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*85)
for transfer, stats in section2_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{transfer:<35} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")


SECTION 2 RESULTS - Fine-tuning
Transfer                            Episodes     Time (s)     Converged    Avg Reward     
-------------------------------------------------------------------------------------
acrobot_to_cartpole                 623          0.00         522          484.87         
cartpole_to_mountaincar             1500         0.00         N/A          -9.41          


## Section 3: Progressive Networks

Transfer from multiple sources:
1. {Acrobot, MountainCar} → CartPole
2. {CartPole, Acrobot} → MountainCar

In [13]:
%%time
# Run or skip Section 3 experiments
if SKIP_SECTION_3:
    print("="*60)
    print("SKIPPING Section 3: Using pre-trained progressive models")
    print("="*60)
    # Create placeholder stats
    section3_results = {
        'acrobot_mountaincar_to_cartpole': TrainingStats(),
        'cartpole_acrobot_to_mountaincar': TrainingStats()
    }
    section3_results['acrobot_mountaincar_to_cartpole'].final_avg_reward = 21.13
    section3_results['acrobot_mountaincar_to_cartpole'].total_episodes = 1500
    section3_results['acrobot_mountaincar_to_cartpole'].training_time = 0
    
    section3_results['cartpole_acrobot_to_mountaincar'].final_avg_reward = -41.61
    section3_results['cartpole_acrobot_to_mountaincar'].total_episodes = 1500
    section3_results['cartpole_acrobot_to_mountaincar'].training_time = 0
    print("Using pre-trained progressive models from repository")
else:
    # Run Section 3 with the FIXED progressive networks implementation!
    section3_results = run_section3_experiments(fast_config)


Section 3 Experiment 1: {Acrobot-v1, MountainCarContinuous-v0} -> CartPole-v1
Loaded source model: Acrobot-v1
Loaded source model: MountainCarContinuous-v0

Progressive Network: ['Acrobot-v1', 'MountainCarContinuous-v0'] -> CartPole-v1
Episode 0, Reward: 10.00, Avg Reward: 10.00, Loss: 18.9253
Model saved to /kaggle/working/DRL-ass3/models/actor_critic_progressive_Acrobot_MountainCarContinuous_to_CartPole.pt
Episode 50, Reward: 19.00, Avg Reward: 22.82, Loss: 18.2533
Episode 100, Reward: 30.00, Avg Reward: 24.38, Loss: 41.7731
Episode 150, Reward: 18.00, Avg Reward: 24.11, Loss: 5.3807
Episode 200, Reward: 20.00, Avg Reward: 21.98, Loss: 3.2137
Model saved to /kaggle/working/DRL-ass3/models/actor_critic_progressive_Acrobot_MountainCarContinuous_to_CartPole.pt
Episode 250, Reward: 11.00, Avg Reward: 20.59, Loss: 1.8538
Episode 300, Reward: 10.00, Avg Reward: 20.30, Loss: 4.5777
Episode 350, Reward: 29.00, Avg Reward: 20.39, Loss: 42.7484
Episode 400, Reward: 18.00, Avg Reward: 23.50, L

KeyboardInterrupt: 

In [ ]:
# Section 3 Summary
print("\n" + "="*70)
print("SECTION 3 RESULTS - Progressive Networks")
print("="*70)
print(f"{'Transfer':<40} {'Episodes':<12} {'Time (s)':<12} {'Converged':<12} {'Avg Reward':<15}")
print("-"*90)
for transfer, stats in section3_results.items():
    conv = stats.convergence_episode if stats.convergence_episode else "N/A"
    print(f"{transfer:<40} {stats.total_episodes:<12} {stats.training_time:<12.2f} {str(conv):<12} {stats.final_avg_reward:<15.2f}")

## Final Comparison

In [ ]:
# Compare results across all sections
print("\n" + "="*90)
print("COMPLETE RESULTS COMPARISON")
print("="*90)

# Compare CartPole training
print("\n--- CartPole Training Comparison ---")
print(f"{'Method':<45} {'Episodes':<12} {'Time (s)':<12}")
print("-"*70)
print(f"{'Section 1: From scratch':<45} {cartpole_stats.total_episodes:<12} {cartpole_stats.training_time:<12.2f}")
print(f"{'Section 2: Fine-tuned from Acrobot':<45} {section2_results['acrobot_to_cartpole'].total_episodes:<12} {section2_results['acrobot_to_cartpole'].training_time:<12.2f}")
print(f"{'Section 3: Progressive (Acrobot+MountainCar)':<45} {section3_results['acrobot_mountaincar_to_cartpole'].total_episodes:<12} {section3_results['acrobot_mountaincar_to_cartpole'].training_time:<12.2f}")

# Compare MountainCar training
print("\n--- MountainCar Training Comparison ---")
print(f"{'Method':<45} {'Episodes':<12} {'Time (s)':<12}")
print("-"*70)
print(f"{'Section 1: From scratch':<45} {mountaincar_stats.total_episodes:<12} {mountaincar_stats.training_time:<12.2f}")
print(f"{'Section 2: Fine-tuned from CartPole':<45} {section2_results['cartpole_to_mountaincar'].total_episodes:<12} {section2_results['cartpole_to_mountaincar'].training_time:<12.2f}")
print(f"{'Section 3: Progressive (CartPole+Acrobot)':<45} {section3_results['cartpole_acrobot_to_mountaincar'].total_episodes:<12} {section3_results['cartpole_acrobot_to_mountaincar'].training_time:<12.2f}")

## TensorBoard Visualization

In [ ]:
# Load TensorBoard
%load_ext tensorboard
%tensorboard --logdir logs/

## Plot Learning Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def moving_avg(data, window=100):
    if len(data) < window:
        return data
    cumsum = np.cumsum(np.insert(data, 0, 0))
    return (cumsum[window:] - cumsum[:-window]) / window

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Section 1 plots
for idx, (env, stats) in enumerate(section1_results.items()):
    ax = axes[0, idx]
    rewards = stats.rewards_history
    ax.plot(rewards, alpha=0.3, label='Episode Reward')
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label='Moving Avg (100)')
    ax.set_title(f'Section 1: {env.split("-")[0]}')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Reward')
    ax.legend()
    ax.grid(True, alpha=0.3)

# Section 2 & 3 comparison for CartPole
ax = axes[1, 0]
for label, stats in [('From Scratch', cartpole_stats),
                      ('Fine-tuned', section2_results['acrobot_to_cartpole']),
                      ('Progressive', section3_results['acrobot_mountaincar_to_cartpole'])]:
    rewards = stats.rewards_history
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label=label)
ax.set_title('CartPole: Transfer Learning Comparison')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (Moving Avg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Section 2 & 3 comparison for MountainCar
ax = axes[1, 1]
for label, stats in [('From Scratch', mountaincar_stats),
                      ('Fine-tuned', section2_results['cartpole_to_mountaincar']),
                      ('Progressive', section3_results['cartpole_acrobot_to_mountaincar'])]:
    rewards = stats.rewards_history
    if len(rewards) >= 100:
        ax.plot(range(99, len(rewards)), moving_avg(rewards), label=label)
ax.set_title('MountainCar: Transfer Learning Comparison')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (Moving Avg)')
ax.legend()
ax.grid(True, alpha=0.3)

# Training time comparison
ax = axes[1, 2]
methods = ['Scratch', 'Fine-tune', 'Progressive']
cartpole_times = [cartpole_stats.training_time,
                  section2_results['acrobot_to_cartpole'].training_time,
                  section3_results['acrobot_mountaincar_to_cartpole'].training_time]
mountaincar_times = [mountaincar_stats.training_time,
                     section2_results['cartpole_to_mountaincar'].training_time,
                     section3_results['cartpole_acrobot_to_mountaincar'].training_time]

x = np.arange(len(methods))
width = 0.35
ax.bar(x - width/2, cartpole_times, width, label='CartPole')
ax.bar(x + width/2, mountaincar_times, width, label='MountainCar')
ax.set_ylabel('Training Time (s)')
ax.set_title('Training Time Comparison')
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('report/learning_curves.png', dpi=150)
plt.show()

## Save Results

In [ ]:
import json
import os
from datetime import datetime

# Create report directory if it doesn't exist
os.makedirs('report', exist_ok=True)

def stats_to_dict(stats):
    return {
        'total_episodes': stats.total_episodes,
        'total_steps': stats.total_steps,
        'training_time': stats.training_time,
        'final_avg_reward': stats.final_avg_reward,
        'convergence_episode': stats.convergence_episode
    }

results = {
    'timestamp': datetime.now().isoformat(),
}

# Add section 1 if exists
if 'section1_results' in dir():
    results['section1'] = {env: stats_to_dict(s) for env, s in section1_results.items()}

# Add section 2 if exists
if 'section2_results' in dir():
    results['section2'] = {k: stats_to_dict(s) for k, s in section2_results.items()}

# Add section 3 if exists
if 'section3_results' in dir():
    results['section3'] = {k: stats_to_dict(s) for k, s in section3_results.items()}

with open('report/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to report/results.json")
print(f"Sections saved: {list(results.keys())}")

In [ ]:
# Create downloadable zip in /kaggle/working/ root (appears in Output tab)
import os

# Create zip with all results
!zip -r /kaggle/working/results.zip models/ logs/ report/ 2>/dev/null || true

# Also copy individual folders to working dir for easy access
!cp -r models/ /kaggle/working/ 2>/dev/null || true
!cp -r logs/ /kaggle/working/ 2>/dev/null || true
!cp -r report/ /kaggle/working/ 2>/dev/null || true

print("Done! Download 'results.zip' from the Output tab on Kaggle")
print("Or click 'Save Version' -> 'Quick Save' then check Output tab")